# 06 — Downstream Classification Utility

## Objective

Evaluate whether synthetic student engagement data preserves
useful predictive information.

XGBoost classifiers are trained exclusively on synthetic data
and evaluated on the real held-out test dataset.

## Experimental configurations

1. CTGAN — Random missingness restoration with target.
2. TVAE — Random missingness restoration with target.
3. CTGAN — From-column missingness restoration with target.
4. TVAE — From-column missingness restoration with target.

## Evaluation metrics

- Weighted precision
- Weighted recall
- Weighted F1-score
- Per-class precision, recall and F1-score
- Confusion matrix

The real test dataset retains its original missing values.
No test data is used during syntheticimport json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from engagement_twin.classifier import (
    build_model,
    train,
    evaluate,
) model training.

In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from engagement_twin.classifier import (
    build_model,
    train,
    evaluate,
)

Configuration

In [2]:
FEATURES = [
    "nevents",
    "ndays_act",
    "ncontent",
    "nforum_posts",
]

TARGET = "Engagement_Level"

SEED = 42

EXPERIMENTS = {
    "random_target": {
        "description": "Random missingness restoration with target",
    },
    "from_column_target": {
        "description": "From-column missingness restoration with target",
    },
}

GENERATORS = ["ctgan", "tvae"]

XGB_PARAMS = {
    "n_estimators": 200,
    "max_depth": 4,
    "learning_rate": 0.1,
    "eval_metric": "logloss",
}

Project paths

In [3]:
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()

    for folder in [start, *start.parents]:
        if (folder / "configs" / "base.yaml").exists():
            return folder

    raise FileNotFoundError(
        "Could not locate project root containing configs/base.yaml"
    )


ROOT = find_project_root()

DATA_DIR = ROOT / "data" / "processed"
GEN_DIR = ROOT / "artifacts" / "generators"
REPORT_DIR = ROOT / "reports"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Training data:", TRAIN_PATH)
print("Test data:", TEST_PATH)
print("Synthetic data:", GEN_DIR)
print("Reports:", REPORT_DIR)

Project root: D:\student_engagement_twin
Training data: D:\student_engagement_twin\data\processed\train.csv
Test data: D:\student_engagement_twin\data\processed\test.csv
Synthetic data: D:\student_engagement_twin\artifacts\generators
Reports: D:\student_engagement_twin\reports


#### Load and validate the datasets

Load real training and test data

In [4]:
if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Training file not found: {TRAIN_PATH}")

if not TEST_PATH.exists():
    raise FileNotFoundError(f"Test file not found: {TEST_PATH}")

real_train = pd.read_csv(TRAIN_PATH)
real_test = pd.read_csv(TEST_PATH)

print("Real training shape:", real_train.shape)
print("Real test shape:", real_test.shape)

display(real_train.head())
display(real_test.head())

Real training shape: (21328, 5)
Real test shape: (5333, 5)


,nevents,ndays_act,ncontent,nforum_posts,Engagement_Level
0,105.0,5.0,7.0,1.0,0
1,930.0,13.0,46.0,NaN,0
2,NaN,1.0,100.0,NaN,0
3,4059.0,8.0,100.0,NaN,1
4,29.0,3.0,7.0,NaN,0


,nevents,ndays_act,ncontent,nforum_posts,Engagement_Level
0,11.0,1.0,7.0,NaN,0
1,NaN,1.0,50.0,NaN,0
2,854.0,1.0,100.0,4.0,1
3,138.0,9.0,100.0,1.0,0
4,523.0,2.0,100.0,3.0,0


Validate the datasets

In [5]:
required_columns = FEATURES + [TARGET]

for name, df in [
    ("Real training", real_train),
    ("Real test", real_test),
]:
    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{name} is missing columns: {missing_columns}"
        )

    if df.empty:
        raise ValueError(f"{name} is empty.")

    if df[TARGET].isna().any():
        raise ValueError(f"{name} contains missing target values.")

    if not df[TARGET].isin([0, 1]).all():
        raise ValueError(f"{name} contains invalid target labels.")

    if df[TARGET].nunique() != 2:
        raise ValueError(f"{name} must contain both target classes.")

print("Both datasets passed basic validation.")

Both datasets passed basic validation.


Inspect missingness and class distribution

In [6]:
print("REAL TRAINING DATA")
print("=" * 50)

display(
    real_train[FEATURES].isna().sum().to_frame(
        "Missing values"
    )
)

print("\nREAL TEST DATA")
print("=" * 50)

display(
    real_test[FEATURES].isna().sum().to_frame(
        "Missing values"
    )
)

print("\nTRAINING CLASS DISTRIBUTION")

display(
    real_train[TARGET]
    .value_counts()
    .sort_index()
    .to_frame("Count")
)

print("\nTEST CLASS DISTRIBUTION")

display(
    real_test[TARGET]
    .value_counts()
    .sort_index()
    .to_frame("Count")
)

REAL TRAINING DATA


,Missing values
nevents,3643
ndays_act,2246
ncontent,0
nforum_posts,14526



REAL TEST DATA


,Missing values
nevents,912
ndays_act,538
ncontent,0
nforum_posts,3618



TRAINING CLASS DISTRIBUTION


,Count
Engagement_Level,
0,17587
1,3741



TEST CLASS DISTRIBUTION


,Count
Engagement_Level,
0,4398
1,935


Confirm that test data retains genuine missing values

In [7]:
train_missing = real_train[FEATURES].isna().sum()
test_missing = real_test[FEATURES].isna().sum()

missingness_comparison = pd.DataFrame({
    "Train Missing": train_missing,
    "Test Missing": test_missing,
    "Train Missing (%)": (
        real_train[FEATURES].isna().mean() * 100
    ).round(2),
    "Test Missing (%)": (
        real_test[FEATURES].isna().mean() * 100
    ).round(2),
})

display(missingness_comparison)

print("\nTest dataset retains its original missing values.")

,Train Missing,Test Missing,Train Missing (%),Test Missing (%)
nevents,3643,912,17.08,17.10
ndays_act,2246,538,10.53,10.09
ncontent,0,0,0.00,0.00
nforum_posts,14526,3618,68.11,67.84



Test dataset retains its original missing values.


#### Load the four synthetic datasets

Load synthetic datasets

In [8]:
synthetic_data = {}

for variant in EXPERIMENTS:

    for generator in GENERATORS:

        filename = f"{generator}_{variant}_restored.csv"
        file_path = GEN_DIR / filename

        if not file_path.exists():
            raise FileNotFoundError(
                f"Synthetic dataset not found: {file_path}"
            )

        df = pd.read_csv(file_path)

        synthetic_data[(variant, generator)] = df

        print(
            f"{variant} | {generator.upper()} | "
            f"Shape: {df.shape}"
        )

random_target | CTGAN | Shape: (21328, 5)
random_target | TVAE | Shape: (21328, 5)
from_column_target | CTGAN | Shape: (21328, 5)
from_column_target | TVAE | Shape: (21328, 5)


Validate synthetic datasets

In [9]:
required_columns = FEATURES + [TARGET]

for (variant, generator), df in synthetic_data.items():

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{variant}/{generator}: missing columns "
            f"{missing_columns}"
        )

    if df.empty:
        raise ValueError(
            f"{variant}/{generator}: dataset is empty."
        )

    if df[TARGET].isna().any():
        raise ValueError(
            f"{variant}/{generator}: missing target values."
        )

    if not df[TARGET].isin([0, 1]).all():
        raise ValueError(
            f"{variant}/{generator}: invalid target labels."
        )

    if df[TARGET].nunique() != 2:
        raise ValueError(
            f"{variant}/{generator}: both classes are required."
        )

print("All four synthetic datasets passed validation.")

All four synthetic datasets passed validation.


Inspect synthetic class distributions

In [10]:
class_distribution = []

for (variant, generator), df in synthetic_data.items():

    counts = df[TARGET].value_counts().sort_index()
    proportions = df[TARGET].value_counts(normalize=True).sort_index()

    class_distribution.append({
        "Variant": variant,
        "Generator": generator.upper(),
        "Class 0 Count": counts.get(0, 0),
        "Class 1 Count": counts.get(1, 0),
        "Class 0 (%)": proportions.get(0, 0) * 100,
        "Class 1 (%)": proportions.get(1, 0) * 100,
    })

class_distribution_df = pd.DataFrame(class_distribution)

display(class_distribution_df.round(2))

,Variant,Generator,Class 0 Count,Class 1 Count,Class 0 (%),Class 1 (%)
0,random_target,CTGAN,17574,3754,82.40,17.60
1,random_target,TVAE,17991,3337,84.35,15.65
2,from_column_target,CTGAN,17814,3514,83.52,16.48
3,from_column_target,TVAE,17693,3635,82.96,17.04


Train and evaluate all four models

In [11]:
classification_results = {}

for (variant, generator), synthetic_df in synthetic_data.items():

    experiment_name = f"{generator}_{variant}"

    print("=" * 65)
    print(f"EXPERIMENT: {experiment_name.upper()}")
    print("=" * 65)

    # Create a fresh model for every experiment
    model = build_model(
        xgb_params=XGB_PARAMS,
        seed=SEED,
    )

    # Train exclusively on synthetic data
    model = train(
        model=model,
        train_df=synthetic_df,
        features=FEATURES,
        target=TARGET,
    )

    # Evaluate on the same untouched real test data
    result = evaluate(
        model=model,
        test=real_test,
        features=FEATURES,
        target=TARGET,
    )

    classification_results[experiment_name] = result

    print("\nWeighted metrics:")

    for metric, score in result["weighted"].items():
        print(f"{metric}: {score:.4f}")

    print("\nPer-class metrics:")

    for label, metrics in result["per_class"].items():
        print(
            f"Class {label}: "
            f"Precision={metrics['precision']:.4f}, "
            f"Recall={metrics['recall']:.4f}, "
            f"F1={metrics['f1-score']:.4f}"
        )

    print("\nConfusion matrix:")
    print(np.array(result["confusion_matrix"]))

EXPERIMENT: CTGAN_RANDOM_TARGET

Weighted metrics:
precision: 0.8665
recall: 0.8766
f1: 0.8669

Per-class metrics:
Class 0: Precision=0.8982, Recall=0.9591, F1=0.9276
Class 1: Precision=0.7174, Recall=0.4888, F1=0.5814

Confusion matrix:
[[4218  180]
 [ 478  457]]
EXPERIMENT: TVAE_RANDOM_TARGET

Weighted metrics:
precision: 0.8944
recall: 0.8995
f1: 0.8956

Per-class metrics:
Class 0: Precision=0.9233, Recall=0.9577, F1=0.9402
Class 1: Precision=0.7588, Recall=0.6257, F1=0.6858

Confusion matrix:
[[4212  186]
 [ 350  585]]
EXPERIMENT: CTGAN_FROM_COLUMN_TARGET

Weighted metrics:
precision: 0.8936
recall: 0.8987
f1: 0.8948

Per-class metrics:
Class 0: Precision=0.9228, Recall=0.9573, F1=0.9397
Class 1: Precision=0.7562, Recall=0.6235, F1=0.6835

Confusion matrix:
[[4210  188]
 [ 352  583]]
EXPERIMENT: TVAE_FROM_COLUMN_TARGET

Weighted metrics:
precision: 0.8972
recall: 0.8952
f1: 0.8961

Per-class metrics:
Class 0: Precision=0.9408, Recall=0.9316, F1=0.9361
Class 1: Precision=0.6922, Rec

Create the performance comparison table

In [12]:
comparison_rows = []

for experiment_name, result in classification_results.items():

    weighted = result["weighted"]

    class_0 = result["per_class"]["0"]
    class_1 = result["per_class"]["1"]

    comparison_rows.append({
        "Experiment": experiment_name,
        "Weighted Precision": weighted["precision"],
        "Weighted Recall": weighted["recall"],
        "Weighted F1": weighted["f1"],
        "Class 0 Recall": class_0["recall"],
        "Class 1 Recall": class_1["recall"],
        "Class 1 F1": class_1["f1-score"],
    })

classification_summary = pd.DataFrame(comparison_rows)

display(
    classification_summary.round(4)
)

,Experiment,Weighted Precision,Weighted Recall,Weighted F1,Class 0 Recall,Class 1 Recall,Class 1 F1
0,ctgan_random_target,0.8665,0.8766,0.8669,0.9591,0.4888,0.5814
1,tvae_random_target,0.8944,0.8995,0.8956,0.9577,0.6257,0.6858
2,ctgan_from_column_target,0.8936,0.8987,0.8948,0.9573,0.6235,0.6835
3,tvae_from_column_target,0.8972,0.8952,0.8961,0.9316,0.7241,0.7078


Real-data baseline

In [13]:
real_baseline_model = build_model(
    xgb_params=XGB_PARAMS,
    seed=SEED,
)

real_baseline_model = train(
    model=real_baseline_model,
    train_df=real_train,
    features=FEATURES,
    target=TARGET,
)

real_baseline_result = evaluate(
    model=real_baseline_model,
    test=real_test,
    features=FEATURES,
    target=TARGET,
)

print("REAL-DATA BASELINE")
print("=" * 60)

print("\nWeighted metrics:")

for metric, score in real_baseline_result["weighted"].items():
    print(f"{metric}: {score:.4f}")

print("\nPer-class metrics:")

for label, metrics in real_baseline_result["per_class"].items():
    print(
        f"Class {label}: "
        f"Precision={metrics['precision']:.4f}, "
        f"Recall={metrics['recall']:.4f}, "
        f"F1={metrics['f1-score']:.4f}"
    )

print("\nConfusion matrix:")
print(np.array(real_baseline_result["confusion_matrix"]))

REAL-DATA BASELINE

Weighted metrics:
precision: 0.9055
recall: 0.9087
f1: 0.9065

Per-class metrics:
Class 0: Precision=0.9345, Recall=0.9563, F1=0.9453
Class 1: Precision=0.7692, Recall=0.6845, F1=0.7244

Confusion matrix:
[[4206  192]
 [ 295  640]]


Real-data baseline

In [14]:
real_baseline_model = build_model(
    xgb_params=XGB_PARAMS,
    seed=SEED,
)

real_baseline_model = train(
    model=real_baseline_model,
    train_df=real_train,
    features=FEATURES,
    target=TARGET,
)

real_baseline_result = evaluate(
    model=real_baseline_model,
    test=real_test,
    features=FEATURES,
    target=TARGET,
)

print("REAL-DATA BASELINE")
print("=" * 60)

print("\nWeighted metrics:")

for metric, score in real_baseline_result["weighted"].items():
    print(f"{metric}: {score:.4f}")

print("\nPer-class metrics:")

for label, metrics in real_baseline_result["per_class"].items():
    print(
        f"Class {label}: "
        f"Precision={metrics['precision']:.4f}, "
        f"Recall={metrics['recall']:.4f}, "
        f"F1={metrics['f1-score']:.4f}"
    )

print("\nConfusion matrix:")
print(np.array(real_baseline_result["confusion_matrix"]))

REAL-DATA BASELINE

Weighted metrics:
precision: 0.9055
recall: 0.9087
f1: 0.9065

Per-class metrics:
Class 0: Precision=0.9345, Recall=0.9563, F1=0.9453
Class 1: Precision=0.7692, Recall=0.6845, F1=0.7244

Confusion matrix:
[[4206  192]
 [ 295  640]]


Utility gap

In [15]:
baseline = real_baseline_result["weighted"]

utility_rows = []

for experiment_name, result in classification_results.items():
    weighted = result["weighted"]

    utility_rows.append({
        "Experiment": experiment_name,
        "Weighted Precision": weighted["precision"],
        "Precision Gap": baseline["precision"] - weighted["precision"],
        "Weighted Recall": weighted["recall"],
        "Recall Gap": baseline["recall"] - weighted["recall"],
        "Weighted F1": weighted["f1"],
        "F1 Gap": baseline["f1"] - weighted["f1"],
        "Class 1 Recall": result["per_class"]["1"]["recall"],
        "Class 1 F1": result["per_class"]["1"]["f1-score"],
    })

utility_comparison = pd.DataFrame(utility_rows)

display(utility_comparison.round(4))

,Experiment,Weighted Precision,Precision Gap,Weighted Recall,Recall Gap,Weighted F1,F1 Gap,Class 1 Recall,Class 1 F1
0,ctgan_random_target,0.8665,0.0390,0.8766,0.0321,0.8669,0.0396,0.4888,0.5814
1,tvae_random_target,0.8944,0.0111,0.8995,0.0092,0.8956,0.0110,0.6257,0.6858
2,ctgan_from_column_target,0.8936,0.0119,0.8987,0.0099,0.8948,0.0117,0.6235,0.6835
3,tvae_from_column_target,0.8972,0.0083,0.8952,0.0135,0.8961,0.0104,0.7241,0.7078


Save results

In [16]:
classification_report = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "test_data": "data/processed/test.csv",
    "training_method": (
        "Synthetic-trained models use synthetic datasets only. "
        "The real-data baseline uses real training data."
    ),
    "evaluation_method": (
        "All models evaluated on the same real held-out test set "
        "with original missing values preserved."
    ),
    "xgboost_parameters": XGB_PARAMS,
    "real_baseline": real_baseline_result,
    "synthetic_experiments": classification_results,
}

report_path = REPORT_DIR / "classification_results.json"

with open(report_path, "w", encoding="utf-8") as file:
    json.dump(classification_report, file, indent=4, allow_nan=False)

print("Classification report saved:", report_path)

Classification report saved: D:\student_engagement_twin\reports\classification_results.json
